# 06 — Baselines and pooling lab

**Project:** Hotel overbooking decision support  
**Stage:** Train-period baselines and Choose-period comparison  
**Purpose:** Establish simple, explainable probability estimates before trying machine-learning models.

A complex model is not useful unless it improves on a sensible baseline. This notebook compares:
1. One cancellation rate for all hotels.
2. A separate cancellation rate for each hotel.
3. A partially pooled hotel rate that balances each hotel's data with the overall rate.

We fit every rate using the **Train** period only. We compare the methods on the **Choose** period. We do not use the **Prove** period in this notebook.

**This is a probability-estimation experiment, not an overbooking policy.** We do not choose how many extra rooms to sell here.

In [1]:
# Run from the repository root, so `from src import ...` and the relative evidence/ paths work
# whether this notebook is opened from the repo folder or from notebooks/pipeline/.
import os
import sys
from pathlib import Path

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "contract.py").exists())
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("Working directory:", REPO_ROOT)

Working directory: C:\Users\Lenovo\Desktop\Build_2_Repo\IIT_GN_W06


## 1. Load the data and contract

The contract defines the chronological periods. Do not change its dates or cost assumptions.

In [2]:
import pandas as pd
from pathlib import Path
from IPython.display import display
from sklearn.metrics import brier_score_loss, log_loss

from src import data, contract

raw = data.read_raw()

bookings = raw.copy()
bookings["arrival_date"] = data.arrival_date(bookings)
bookings["nights"] = (
    bookings["stays_in_weekend_nights"] + bookings["stays_in_week_nights"]
)

train, choose, proof = contract.split_periods(bookings)

print("Train rows:", f"{len(train):,}")
print("Choose rows:", f"{len(choose):,}")
print("Proof rows:", f"{len(proof):,}")

Train rows: 49,157
Choose rows: 29,546
Proof rows: 40,687


## 2. Check the time boundaries

The Train set ends before 1 July 2016. Choose covers 1 July through 31 December 2016. Proof begins on 1 January 2017.

The Proof data must not influence the rates, smoothing strength or method selection.

In [3]:
print("Train:", train["arrival_date"].min().date(), "to", train["arrival_date"].max().date())
print("Choose:", choose["arrival_date"].min().date(), "to", choose["arrival_date"].max().date())
print("Proof:", proof["arrival_date"].min().date(), "to", proof["arrival_date"].max().date())

assert train["arrival_date"].max() < pd.Timestamp(contract.TRAIN_END)
assert choose["arrival_date"].min() >= pd.Timestamp(contract.TRAIN_END)
assert choose["arrival_date"].max() < pd.Timestamp(contract.PROOF_START)
assert proof["arrival_date"].min() >= pd.Timestamp(contract.PROOF_START)

print("Chronological split checks passed.")

Train: 2015-07-01 to 2016-06-30
Choose: 2016-07-01 to 2016-12-31
Proof: 2017-01-01 to 2017-08-31
Chronological split checks passed.


## 3. Establish the overall Train cancellation rate

The global baseline assigns the same probability to every booking. It is simple and provides a reference point for the hotel-specific methods.

In [4]:
global_rate = train["is_canceled"].mean()

print(f"Overall Train cancellation rate: {global_rate:.4f}")
print(f"Percentage: {global_rate:.2%}")

Overall Train cancellation rate: 0.3577
Percentage: 35.77%


## 4. Calculate each hotel's Train rate

A hotel-specific baseline allows the two properties to have different rates.

A potential weakness is that a rate based on a smaller sample can be less stable. We will compare it with partial pooling.

In [5]:
hotel_summary = (
    train.groupby("hotel")["is_canceled"]
    .agg(bookings="size", cancellations="sum", hotel_rate="mean")
    .reset_index()
)

display(hotel_summary)

,hotel,bookings,cancellations,hotel_rate
0,City Hotel,31623,13175,0.416627
1,Resort Hotel,17534,4410,0.251511


## 5. Understand partial pooling

Partial pooling moves a hotel's observed rate partway towards the overall Train rate.

The formula is:

\[
p_h = \frac{c_h + m p_{global}}{n_h + m}
\]

Where:
- \(c_h\) is the number of cancellations for hotel \(h\) in Train;
- \(n_h\) is the number of Train bookings for hotel \(h\);
- \(p_{global}\) is the overall Train cancellation rate;
- \(m\) is the prior strength, measured in booking-equivalents.

A larger \(m\) pulls the hotel rate more strongly towards the global rate. A smaller \(m\) trusts the hotel's own data more.

This is a simple smoothing method, not a claim that the global rate is literally a Bayesian prior distribution.

In [6]:
def pooled_hotel_rates(training_data, prior_strength):
    overall_rate = training_data["is_canceled"].mean()

    summary = (
        training_data.groupby("hotel")["is_canceled"]
        .agg(bookings="size", cancellations="sum")
    )

    pooled_rate = (
        summary["cancellations"] + prior_strength * overall_rate
    ) / (
        summary["bookings"] + prior_strength
    )

    return pooled_rate


prior_strengths = [20, 50, 100]

pooled_table = pd.DataFrame({
    "hotel": hotel_summary["hotel"]
})

for strength in prior_strengths:
    rates = pooled_hotel_rates(train, strength)
    pooled_table[f"pooled_m_{strength}"] = (
        pooled_table["hotel"].map(rates)
    )

display(pooled_table)

,hotel,pooled_m_20,pooled_m_50,pooled_m_100
0,City Hotel,0.416590,0.416534,0.416441
1,Resort Hotel,0.251632,0.251813,0.252114


## 6. Build predictions for the Choose period

All estimates below come only from Train. We now apply those fixed estimates to Choose.

The `global` method uses one rate for both hotels. `hotel_specific` uses each hotel's raw Train rate. The pooled methods use the listed prior strength.

In [7]:
train_hotel_rates = train.groupby("hotel")["is_canceled"].mean()

predictions = pd.DataFrame(index=choose.index)
predictions["actual"] = choose["is_canceled"].astype(int)
predictions["global"] = global_rate
predictions["hotel_specific"] = choose["hotel"].map(train_hotel_rates)

for strength in prior_strengths:
    rates = pooled_hotel_rates(train, strength)
    predictions[f"pooled_m_{strength}"] = choose["hotel"].map(rates)

assert predictions.notna().all().all()
assert predictions.drop(columns="actual").apply(
    lambda column: column.between(0, 1).all()
).all()

display(predictions.head())

,actual,global,hotel_specific,pooled_m_20,pooled_m_50,pooled_m_100
6765,1,0.357731,0.251511,0.251632,0.251813,0.252114
6766,1,0.357731,0.251511,0.251632,0.251813,0.252114
6767,1,0.357731,0.251511,0.251632,0.251813,0.252114
6768,1,0.357731,0.251511,0.251632,0.251813,0.252114
6769,0,0.357731,0.251511,0.251632,0.251813,0.252114


## 7. Compare booking-level probability quality

We use two standard probability metrics:

- **Brier score:** mean squared difference between predicted probability and the actual 0/1 outcome. Lower is better.
- **Log loss:** penalises confident predictions that are wrong. Lower is better.

These metrics assess probability quality. They do not directly measure the financial cost of an overbooking decision.

In [8]:
prediction_columns = [
    "global",
    "hotel_specific",
    "pooled_m_20",
    "pooled_m_50",
    "pooled_m_100"
]

booking_results = []

for method in prediction_columns:
    p = predictions[method]
    booking_results.append({
        "method": method,
        "brier_score": brier_score_loss(predictions["actual"], p),
        "log_loss": log_loss(predictions["actual"], p, labels=[0, 1])
    })

booking_metrics = (
    pd.DataFrame(booking_results)
    .sort_values("brier_score")
    .reset_index(drop=True)
)

display(booking_metrics)

,method,brier_score,log_loss
0,pooled_m_100,0.229853,0.651993
1,pooled_m_50,0.229860,0.652012
2,pooled_m_20,0.229864,0.652023
3,hotel_specific,0.229867,0.652031
4,global,0.232884,0.658529


## 8. Compare expected cancellations at hotel-arrival-night level

The client's decision is made per hotel and arrival night, not per individual booking.

For each method, we add the booking probabilities within each hotel-arrival night. We compare that expected cancellation count with the actual number of cancellations.

This is a useful bridge between booking-level probabilities and the client's operational unit. It is still not an overbooking policy or a room-capacity calculation.

In [9]:
nightly_results = []

for method in prediction_columns:
    scored = choose[["hotel", "arrival_date", "is_canceled"]].copy()
    scored["probability"] = predictions[method].values

    by_night = scored.groupby(["hotel", "arrival_date"]).agg(
        actual_cancellations=("is_canceled", "sum"),
        expected_cancellations=("probability", "sum")
    )

    mae = (
        by_night["actual_cancellations"]
        - by_night["expected_cancellations"]
    ).abs().mean()

    nightly_results.append({
        "method": method,
        "hotel_arrival_nights": len(by_night),
        "mean_absolute_error_cancellations": mae
    })

nightly_metrics = (
    pd.DataFrame(nightly_results)
    .sort_values("mean_absolute_error_cancellations")
    .reset_index(drop=True)
)

display(nightly_metrics)

,method,hotel_arrival_nights,mean_absolute_error_cancellations
0,pooled_m_100,368,9.187536
1,pooled_m_50,368,9.189827
2,pooled_m_20,368,9.191206
3,hotel_specific,368,9.192128
4,global,368,9.661821


## 9. Inspect results separately for each hotel

An overall score can hide differences between the City Hotel and Resort Hotel. This table reports Brier score separately for each property.

In [10]:
hotel_metrics = []

for hotel_name, group in choose.groupby("hotel"):
    hotel_index = group.index
    actual = predictions.loc[hotel_index, "actual"]

    for method in prediction_columns:
        hotel_metrics.append({
            "hotel": hotel_name,
            "method": method,
            "brier_score": brier_score_loss(
                actual,
                predictions.loc[hotel_index, method]
            )
        })

hotel_metrics = pd.DataFrame(hotel_metrics)
display(hotel_metrics.pivot(
    index="method",
    columns="hotel",
    values="brier_score"
))

hotel,City Hotel,Resort Hotel
method,,
global,0.243990,0.208885
hotel_specific,0.241568,0.204583
pooled_m_100,0.241564,0.204544
pooled_m_20,0.241567,0.204575
pooled_m_50,0.241566,0.204563


## 10. Make the baseline decision

Do not automatically choose the method with the smallest score. First consider:
- Is the improvement meaningful or tiny?
- Does one hotel improve while the other gets worse?
- Does a pooled method behave more consistently than the raw hotel rate?
- Is the simplest method close enough to the more complex one?
- Are the data and feature-timing limitations still material?

Record the selected baseline and the reason in `docs/DECISIONS.md`. Do not inspect Proof results to make this choice.

In [11]:
best_booking_method = booking_metrics.iloc[0]["method"]
best_nightly_method = nightly_metrics.iloc[0]["method"]

print("Lowest Choose booking Brier score:", best_booking_method)
print("Lowest Choose nightly cancellation-count MAE:", best_nightly_method)
print()
print("These can be different methods because they measure different things.")
print("Review the per-hotel table and business implications before selecting a baseline.")

Lowest Choose booking Brier score: pooled_m_100
Lowest Choose nightly cancellation-count MAE: pooled_m_100

These can be different methods because they measure different things.
Review the per-hotel table and business implications before selecting a baseline.


## 11. Save the experiment evidence

The saved files contain Train-only rate estimates and Choose-period comparisons. They do not contain Proof-period scores.

In [12]:
evidence_folder = Path("evidence/baselines_pooling")
evidence_folder.mkdir(parents=True, exist_ok=True)

hotel_summary.to_csv(evidence_folder / "train_hotel_rates.csv", index=False)
pooled_table.to_csv(evidence_folder / "train_pooled_rates.csv", index=False)
booking_metrics.to_csv(evidence_folder / "choose_booking_metrics.csv", index=False)
nightly_metrics.to_csv(evidence_folder / "choose_nightly_metrics.csv", index=False)
hotel_metrics.to_csv(evidence_folder / "choose_hotel_metrics.csv", index=False)

print("Saved baseline evidence to:", evidence_folder)

Saved baseline evidence to: evidence\baselines_pooling


## 12. Limitations and questions to answer

### Limitations

- These baselines use only the hotel identifier and historical cancellation outcomes. They do not use the candidate features from Notebook 05.
- The hotel-specific estimates assume the Train rate remains useful in the later Choose period.
- Partial pooling is a simple stabilisation technique; the prior strengths tested here are illustrative candidates.
- Booking-level Brier score and nightly cancellation-count MAE answer different questions.
- No room inventory, capacity, rooms-per-booking or actual walk records are available in this dataset.
- No oversell count is selected in this notebook.

### Questions

1. Why is a global rate a useful baseline?
2. Why might the hotel-specific rate be unstable?
3. What happens as the pooling strength \(m\) increases?
4. Why can the best booking-level Brier score differ from the best nightly cancellation-count MAE?
5. Why must the Proof period remain unused while selecting a method?

**Next notebook:** `07_train_model_experiments.ipynb`, where we fit candidate models using Train and compare them on Choose.